In [1]:
import earthaccess
import getpass
import os
from datetime import datetime
import geopandas as gpd
from shapely.geometry import box, Polygon, mapping
import pandas as pd
import folium
import webbrowser
from tqdm import tqdm
from openpyxl.utils import get_column_letter

In [2]:
# === Paths ===
IRAN_SHP_PATH = r"G:\IRAN\gadm41_IRN_shp\gadm41_IRN_0.shp"
OUTPUT_DIR = r"G:\OUTPUT\IRAN-Coverage"


# ===================================================================
# AUTHENTICATION
# ===================================================================
def authenticate_earthdata():
    username = input("Enter Earthdata username: ").strip()
    password = getpass.getpass("Enter Earthdata password: ")

    try:
        os.environ["EARTHDATA_USERNAME"] = username
        os.environ["EARTHDATA_PASSWORD"] = password

        auth = earthaccess.login(strategy="environment")
        if auth.authenticated:
            print("\nAuthentication successful!")
            print("=" * 70)
            return auth
        else:
            print("Authentication failed.")
            return None

    except Exception as e:
        print(f"Authentication failed: {e}")
        return None


# ===================================================================
# LOAD IRAN SHAPEFILE
# ===================================================================
def load_iran_shapefile():
    print("=" * 70)
    print("Loading Iran shapefile...")
    print("=" * 70)

    try:
        iran_gdf = gpd.read_file(IRAN_SHP_PATH)
        iran_gdf = iran_gdf.to_crs(epsg=4326)
        iran_polygon = iran_gdf.union_all()

        print("Iran shapefile loaded successfully.")
        return iran_gdf, iran_polygon

    except Exception as e:
        print(f"Failed to load Iran shapefile: {e}")
        return None, None


# ===================================================================
# SEARCH MOD021KM (Terra)
# ===================================================================
def search_modis_data(temporal_range, spatial_bbox):

    print(f"\n=== Searching Terra (MOD021KM) ===")
    try:
        results = earthaccess.search_data(
            short_name="MOD021KM",
            temporal=temporal_range,
            bounding_box=spatial_bbox,
        )
        print(f"Found {len(results)} Terra granules.")
        return results

    except Exception as e:
        print(f"Search failed: {e}")
        return []


# ===================================================================
# SEARCH MYD021KM (Aqua)
# ===================================================================
def search_aqua_data(temporal_range, spatial_bbox):

    print(f"\n=== Searching Aqua (MYD021KM) ===")
    try:
        results = earthaccess.search_data(
            short_name="MYD021KM",
            temporal=temporal_range,
            bounding_box=spatial_bbox,
        )
        print(f"Found {len(results)} Aqua granules.")
        return results

    except Exception as e:
        print(f"Search failed: {e}")
        return []


# ===================================================================
# FULLY FIXED & VERIFIED METADATA EXTRACTOR
# ===================================================================
def extract_granule_metadata(granule):
    metadata = {}

    try:
        umm = granule.get("umm", {})

        # ----------------------------
        # 1) PLATFORM (MOD → Terra, MYD → Aqua)
        # ----------------------------
        short_name = umm.get("CollectionReference", {}).get("ShortName", "")
        if "MYD" in short_name:
            metadata["platform"] = "Aqua"
        elif "MOD" in short_name:
            metadata["platform"] = "Terra"
        else:
            metadata["platform"] = "Unknown"

        # ----------------------------
        # 2) TIME
        # ----------------------------
        temporal = umm.get("TemporalExtent", {}).get("RangeDateTime", {})
        metadata["start_time"] = temporal.get("BeginningDateTime", None)
        metadata["end_time"] = temporal.get("EndingDateTime", None)

        # ----------------------------
        # 3) GRANULE ID (اصلاح اصلی اینجاست)
        # ----------------------------
        dg = umm.get("DataGranule", {})

        gran_id = dg.get("ProducerGranuleId")
        if not gran_id:
            gran_id = umm.get("GranuleUR")
        if not gran_id:
            gran_id = umm.get("GranuleURMetaData", {}).get("GranuleUR")

        metadata["producer_granule_id"] = str(gran_id) if gran_id else None

        # ----------------------------
        # 4) SIZE (MB)
        # ----------------------------
        size_mb = None
        dist = dg.get("ArchiveAndDistributionInformation", [{}])[0]

        if "SizeInBytes" in dist:
            size_mb = dist["SizeInBytes"] / (1024 * 1024)
        elif "Size" in dist and dist.get("SizeUnit", "").upper() == "MB":
            size_mb = float(dist["Size"])

        metadata["size_mb"] = round(size_mb, 3) if size_mb else None

        # ----------------------------
        # 5) SPATIAL (BBOX / GPOLYGON)
        # ----------------------------
        geometry = (
            umm.get("SpatialExtent", {})
               .get("HorizontalSpatialDomain", {})
               .get("Geometry", {})
        )

        if "BoundingRectangles" in geometry:
            b = geometry["BoundingRectangles"][0]
            metadata["bbox"] = {
                "west": float(b["WestBoundingCoordinate"]),
                "south": float(b["SouthBoundingCoordinate"]),
                "east": float(b["EastBoundingCoordinate"]),
                "north": float(b["NorthBoundingCoordinate"]),
            }

        elif "GPolygons" in geometry:
            pts = geometry["GPolygons"][0]["Boundary"]["Points"]
            pts = [(float(p["Longitude"]), float(p["Latitude"])) for p in pts]
            metadata["polygon_points"] = pts

            lons = [p[0] for p in pts]
            lats = [p[1] for p in pts]
            metadata["bbox"] = {
                "west": min(lons),
                "south": min(lats),
                "east": max(lons),
                "north": max(lats),
            }

        return metadata

    except Exception as e:
        print("[ERROR] extract_granule_metadata:", e)
        return metadata


# ===================================================================
# CREATE GEOMETRY
# ===================================================================
def create_granule_polygon(metadata):
    try:
        if "polygon_points" in metadata:
            return Polygon(metadata["polygon_points"])

        if "bbox" in metadata:
            b = metadata["bbox"]
            return box(b["west"], b["south"], b["east"], b["north"])

    except Exception as e:
        print("[ERROR] create_granule_polygon:", e)

    return None


# ===================================================================
# FAST BBOX INTERSECT
# ===================================================================
def quick_bbox_intersect(metadata, iran_bbox):
    if "bbox" not in metadata:
        return False

    b = metadata["bbox"]
    gran_box = box(b["west"], b["south"], b["east"], b["north"])
    return gran_box.intersects(iran_bbox)


# ===================================================================
# PREFILTER USING IRAN BBOX
# ===================================================================
def pre_filter_granules(results, iran_bbox):
    out = []
    for gran in tqdm(results, desc="Pre-filtering"):
        m = extract_granule_metadata(gran)
        if quick_bbox_intersect(m, iran_bbox):
            gran.__dict__["metadata"] = m
            out.append(gran)

    print(f"Kept {len(out)} / {len(results)} after bbox filtering.")
    return out


# ===================================================================
# FULL COVERAGE ANALYSIS
# ===================================================================
def analyze_granules(results, iran_poly):
    final = []

    for gran in tqdm(results, desc="Coverage analysis"):
        m = gran.__dict__.get("metadata", extract_granule_metadata(gran))
        poly = create_granule_polygon(m)

        if poly and poly.intersects(iran_poly):
            inter = poly.intersection(iran_poly)
            m["coverage_percentage"] = round(
                (inter.area / iran_poly.area) * 100, 2
            )
            m["geometry"] = poly
            final.append(m)

    print(f"Found {len(final)} granules with coverage > 0%.")
    return final


# ===================================================================
# EXPORT HTML
# ===================================================================
def export_to_html_map(granules, iran_gdf, out, temporal_range):

    start = temporal_range[0].replace("-", ".")
    end = temporal_range[1].replace("-", ".")
    name = f"MODIS_Iran_Coverage_Map_{start}_{end}.html"
    path = os.path.join(out, name)

    m = folium.Map(location=[32, 53], zoom_start=5)

    folium.GeoJson(
        iran_gdf,
        style_function=lambda x: {
            "color": "darkgreen",
            "weight": 3,
            "fillOpacity": 0.1,
        },
    ).add_to(m)

    for g in granules:
        poly = g["geometry"]
        cov = g["coverage_percentage"]

        color = (
            "purple"
            if cov >= 90
            else "red"
            if cov >= 80
            else "orange"
            if cov >= 70
            else "yellow"
            if cov >= 50
            else "blue"
        )

        folium.GeoJson(
            mapping(poly),
            style_function=lambda x, c=color: {
                "color": c,
                "fillColor": c,
                "fillOpacity": 0.3,
            },
        ).add_to(m)

    m.save(path)
    print(f"HTML saved → {path}")


# ===================================================================
# EXPORT EXCEL
# ===================================================================
def export_to_excel(granules, out, temporal_range):

    start = temporal_range[0].replace("-", ".")
    end = temporal_range[1].replace("-", ".")
    path = os.path.join(out, f"MODIS_Iran_Results_{start}_{end}.xlsx")

    df = pd.DataFrame(granules)

    df = df[df["coverage_percentage"] >= 50]
    df = df.sort_values("coverage_percentage", ascending=False)

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(
            writer,
            sheet_name="MODIS Granules",
            index=False,
            columns=[
                "platform",
                "producer_granule_id",
                "start_time",
                "end_time",
                "size_mb",
                "coverage_percentage",
            ],
        )

        sheet = writer.sheets["MODIS Granules"]
        for i, col in enumerate(df.columns, 1):
            sheet.column_dimensions[get_column_letter(i)].width = 22

    print(f"EXCEL saved → {path}")


# ===================================================================
# MAIN
# ===================================================================
def main():

    authenticate_earthdata()

    temporal_range = ("2018-01-01", "2025-11-15")
    bbox = (44, 25, 63.5, 40)

    terra = search_modis_data(temporal_range, bbox)
    aqua = search_aqua_data(temporal_range, bbox)
    results = terra + aqua

    iran_gdf, iran_poly = load_iran_shapefile()
    iran_bbox = box(*iran_poly.bounds)

    filtered_terra = pre_filter_granules(terra, iran_bbox)
    filtered_aqua = pre_filter_granules(aqua, iran_bbox)

    analy_terra = analyze_granules(filtered_terra, iran_poly)
    analy_aqua = analyze_granules(filtered_aqua, iran_poly)

    all_results = analy_terra + analy_aqua

    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"Terra found: {len(terra)}")
    print(f"Aqua  found: {len(aqua)}")
    print(f"After bbox filter: {len(filtered_terra)+len(filtered_aqua)}")
    print(f"After coverage: {len(all_results)}")

    export_to_html_map(all_results, iran_gdf, OUTPUT_DIR, temporal_range)
    export_to_excel(all_results, OUTPUT_DIR, temporal_range)


if __name__ == "__main__":
    main()

Enter Earthdata username:  1995Mnona
Enter Earthdata password:  ········



Authentication successful!

=== Searching Terra (MOD021KM) ===
Found 21342 Terra granules.

=== Searching Aqua (MYD021KM) ===
Found 21301 Aqua granules.
Loading Iran shapefile...
Iran shapefile loaded successfully.


Pre-filtering: 100%|██████████████████████████████████████████████████████████| 21342/21342 [00:01<00:00, 15842.67it/s]


Kept 21253 / 21342 after bbox filtering.


Pre-filtering: 100%|██████████████████████████████████████████████████████████| 21301/21301 [00:00<00:00, 28558.77it/s]


Kept 21174 / 21301 after bbox filtering.


Coverage analysis: 100%|█████████████████████████████████████████████████████████| 21253/21253 [44:53<00:00,  7.89it/s]


Found 18277 granules with coverage > 0%.


Coverage analysis: 100%|█████████████████████████████████████████████████████████| 21174/21174 [44:23<00:00,  7.95it/s]


Found 18105 granules with coverage > 0%.
SUMMARY
Terra found: 21342
Aqua  found: 21301
After bbox filter: 42427
After coverage: 36382
HTML saved → G:\OUTPUT\IRAN-Coverage\MODIS_Iran_Coverage_Map_2018.01.01_2025.11.15.html
EXCEL saved → G:\OUTPUT\IRAN-Coverage\MODIS_Iran_Results_2018.01.01_2025.11.15.xlsx
